In [7]:
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data_loading import load_csv_files, build_image_path
from src.preprocessing import ImageClassificationDataset, get_eval_transforms
from src.inference import load_model, predict_labels, build_submission, save_submission
from src.training import get_device

OUTPUT_DIR = ROOT / 'outputs'
MODEL_PATH = OUTPUT_DIR / 'baseline_cnn.pt'
SUBMISSION_PATH = OUTPUT_DIR / 'submission.csv'

print('model exists:', MODEL_PATH.exists())
print('output directory:', OUTPUT_DIR)

model exists: True
output directory: /Users/Valeria/Desktop/Image & Video Processing /iivp-2026-challenge/outputs


In [8]:
# Loading test data and building the test dataloader.
# Testing data has no labels so we set labels=None." 
test_df = load_csv_files()[1]
test_transform = get_eval_transforms(32)
test_paths = [build_image_path(image_id, split='test') for image_id in test_df['Id'].tolist()]

test_dataset = ImageClassificationDataset(test_paths, labels=None, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print('test samples:', len(test_dataset))
print('first path:', test_paths[0])

test samples: 3000
first path: /Users/Valeria/Desktop/Image & Video Processing /iivp-2026-challenge/test/test/56604.png


In [9]:
# Loading the trained model and generating predictions on the test set.
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model checkpoint not found: {MODEL_PATH}. Run training_and_validation.ipynb first to save it.')

device = get_device()
model = load_model(MODEL_PATH, device)
predicted_labels = predict_labels(model, test_loader, device)

# Building the submission CSV with image ids and predicted labels.
submission_df = build_submission(test_df['Id'].tolist(), predicted_labels)
save_submission(submission_df, SUBMISSION_PATH)

print('saved submission to:', SUBMISSION_PATH)
print(submission_df.head())

saved submission to: /Users/Valeria/Desktop/Image & Video Processing /iivp-2026-challenge/outputs/submission.csv
      Id  Category
0  56604         6
1  29396         3
2  43803         6
3  12313         0
4  10341         8
